In [13]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from variable_mappings import GENDER_MAP, AGE_MAP, RACE_MAP, EDUCATION_MAP, INCOME_MAP, RELATIONSHIP_MAP, CHAT_USAGE_MAP, TENURE_MAP

In [3]:
df = pd.read_csv('../data/participant_data.csv')

## Report Basic Metadata

In [10]:
print("Total number of participants: ", len(df))
treatment_group = df[df['baseline_treatment'] == 1]
control_group = df[df['baseline_treatment'] == 0]
print("Number of participants in the treatment group: ", len(treatment_group))
print("Number of participants in the control group: ", len(control_group))
print("Percentage of participants in the treatment group: ", len(treatment_group) / len(df))
print("Percentage of participants in the control group: ", len(control_group) / len(df))
completed_endline = df[df['completed_endline'] == 1]
completed_post = df[df['completed_post'] == 1]
print("Number of participants who completed the endline: ", len(completed_endline))
print("Percentage of participants who completed the endline: ", len(completed_endline) / len(df))
print("Number of participants who completed the post: ", len(completed_post))
print("Percentage of participants who completed the post (of those that completed the endline): ", len(completed_post) / len(completed_endline))



Total number of participants:  625
Number of participants in the treatment group:  306
Number of participants in the control group:  319
Percentage of participants in the treatment group:  0.4896
Percentage of participants in the control group:  0.5104
Number of participants who completed the endline:  500
Percentage of participants who completed the endline:  0.8
Number of participants who completed the post:  442
Percentage of participants who completed the post (of those that completed the endline):  0.884


## Chi-Squared Analysis of Participant Demographics

In [16]:
rng = np.random.default_rng(0)
N_PERM = 10000
df["condition"] = df["baseline_treatment"].astype(bool).map({True: "Treatment", False: "Control"})

def invert(m):
    out = {}
    for k, v in m.items():
        out.setdefault(v, k)
    return out

VARS = {  # display name -> (column, code -> label map)
    "Age": ("baseline_Age", invert(AGE_MAP)),
    "Gender": ("baseline_Gender", invert(GENDER_MAP)),
    "Race": ("baseline_Race", invert(RACE_MAP)),
    "Education": ("baseline_EdLevel", invert(EDUCATION_MAP)),
    "Income": ("baseline_Income", invert(INCOME_MAP)),
    "Relationship status": ("baseline_Relationship", invert(RELATIONSHIP_MAP)),
    "AI chatbot daily use": ("Usage", invert(CHAT_USAGE_MAP)),
    "AI chatbot tenure": ("Tenure", invert(TENURE_MAP)),
}
for name, (col, labels) in VARS.items():
    df[name] = pd.Categorical(df[col].map(labels), categories=list(dict.fromkeys(labels[k] for k in sorted(labels))), ordered=True)
    df[name] = df[name].cat.remove_unused_categories()

In [17]:
def chi2_test(data, var, n_perm=N_PERM):
    sub = data[[var, "condition"]].dropna()
    tab = pd.crosstab(sub[var], sub["condition"])
    tab = tab.loc[tab.sum(axis=1) > 0]
    chi2, p, dof, expected = chi2_contingency(tab, correction=False)
    # Monte Carlo p: permute condition labels, recompute the statistic
    # (vectorized: margins are fixed under permutation, so expected counts stay the same)
    x = pd.Categorical(sub[var]).remove_unused_categories().codes
    g = (sub["condition"] == "Treatment").to_numpy().astype(int)
    k, E = x.max() + 1, expected
    perm_g = np.array([rng.permutation(g) for _ in range(n_perm)])          # (n_perm, n)
    treat = np.stack([(perm_g * (x == j)).sum(axis=1) for j in range(k)], axis=1)  # (n_perm, k)
    row_tot = np.bincount(x, minlength=k)
    obs = np.stack([row_tot - treat, treat], axis=2)                          # (n_perm, k, 2); cols Control, Treatment
    perm = ((obs - E) ** 2 / E).sum(axis=(1, 2))
    p_mc = (1 + (perm >= chi2 - 1e-12).sum()) / (n_perm + 1)
    return {"variable": var, "n": len(sub), "chi2": chi2, "df": dof, "p": p, "p_mc": p_mc,
            "pct_cells_exp_lt5": 100 * (expected < 5).mean(), "min_expected": expected.min()}

samples = {"Randomized": df, "Completed endline": df[df.completed_endline]}
results = pd.concat({s: pd.DataFrame([chi2_test(d, v) for v in VARS]) for s, d in samples.items()}, names=["sample"]).reset_index(level=0)
results.round(3)

,sample,variable,n,chi2,df,p,p_mc,pct_cells_exp_lt5,min_expected
0,Randomized,Age,625,2.523,5,0.773,0.779,0.000,7.344
1,Randomized,Gender,625,2.362,3,0.501,0.512,50.000,0.490
2,Randomized,Race,625,4.101,7,0.768,0.810,37.500,0.979
3,Randomized,Education,625,10.078,6,0.121,0.089,28.571,0.490
4,Randomized,Income,625,6.118,6,0.410,0.406,14.286,3.427
5,Randomized,Relationship status,625,8.530,7,0.288,0.298,37.500,1.958
6,Randomized,AI chatbot daily use,624,1.265,3,0.737,0.726,0.000,25.500
7,Randomized,AI chatbot tenure,624,1.226,3,0.747,0.744,0.000,27.462
0,Completed endline,Age,500,2.811,5,0.729,0.732,0.000,5.214
1,Completed endline,Gender,500,1.851,2,0.396,0.387,33.333,1.896
